# 📱 Liquidity Stress Prediction Challenge - Starter Notebook

Welcome to the my notebook for this challenge.

In this challenge, my goal is to build a machine learning model that predicts whether a customer is likely to experience liquidity stress, using historical mobile money behaviour.

This notebook walks through a simple end-to-end pipeline:

- Loading and exploring the data
- Preprocessing
- Training a baseline model
- Evaluating performance
- Generating predictions for submission

The baseline model used here is **Logistic Regression**.


In [1]:
# 📚 Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)


In [2]:
# 📥 Load the Data
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

# Peek at the data
train.head()


,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highe

## Data cleaning and exploration

Here we inspect the data for important information that could affect the training process and model building.

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Columns: 184 entries, arpu to ID
dtypes: float64(134), int64(44), str(6)
memory usage: 56.2 MB


In [4]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Columns: 183 entries, arpu to ID
dtypes: float64(134), int64(43), str(6)
memory usage: 41.9 MB


In [5]:
# Check missing values
missing_summary = train.isnull().sum().sort_values(ascending=False)
# missing_summary[missing_summary > 0].head(20)
missing_summary ## There are no missing values

arpu                         0
age                          0
gender                       0
region                       0
smartphone                   0
                            ..
m4_daily_avg_bal             0
m5_daily_avg_bal             0
m6_daily_avg_bal             0
liquidity_stress_next_30d    0
ID                           0
Length: 184, dtype: int64

In [6]:
missing_summary[missing_summary > 0]

Series([], dtype: int64)

There are no missing values.

### 🏷️ Encoding Categorical Variables

We use **one-hot encoding** to convert categorical columns into numeric format so they can be used by the model.


In [7]:
# Identify the ID column, target column and feature columns
TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"
object_type=['str','object']

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET]]
cat_cols = [c for c in feature_cols if train[c].dtype in object_type]
num_cols = (
    train.select_dtypes(include=['number'])
    .drop(columns=[TARGET], errors='ignore')
    .columns.tolist()
)

cat_cols


['gender', 'region', 'smartphone', 'segment', 'earning_pattern']

In [8]:
## Inspect categorical columns for misspelling
for col in cat_cols:
  print(f"Unique values in {col}:", train[col].unique())

Unique values in gender: <StringArray>
['F', 'M']
Length: 2, dtype: str
Unique values in region: <StringArray>
[              'Central Zone',         'Northern Highlands',
      'Lake Zone (Northwest)', 'Coastal & Eastern Corridor',
         'Southern Highlands',                   'Islander',
               'Western Zone']
Length: 7, dtype: str
Unique values in smartphone: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Unique values in segment: <StringArray>
['MVC', 'LVC', 'HVC']
Length: 3, dtype: str
Unique values in earning_pattern: <StringArray>
['Monthly Earner', 'Weekly Earner', 'Irregular Earner']
Length: 3, dtype: str


In [9]:
for col in cat_cols:
  print(f"Unique values in {col}:", test[col].unique())

Unique values in gender: <StringArray>
['F', 'M']
Length: 2, dtype: str
Unique values in region: <StringArray>
[              'Central Zone',         'Northern Highlands',
      'Lake Zone (Northwest)', 'Coastal & Eastern Corridor',
         'Southern Highlands',                   'Islander',
               'Western Zone']
Length: 7, dtype: str
Unique values in smartphone: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Unique values in segment: <StringArray>
['MVC', 'LVC', 'HVC']
Length: 3, dtype: str
Unique values in earning_pattern: <StringArray>
['Monthly Earner', 'Weekly Earner', 'Irregular Earner']
Length: 3, dtype: str


In [10]:
## Split to Features and Target
X=train[feature_cols]
y=train[TARGET]

Split the training data to training and validation. We apply statify to ensure that equal proportion of the 1s and 0s are accounted for within each split

In [11]:
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

In [12]:
y_train.value_counts()

liquidity_stress_next_30d
0    23800
1     4200
Name: count, dtype: int64

In [13]:
y_val.value_counts()

liquidity_stress_next_30d
0    10200
1     1800
Name: count, dtype: int64

In [14]:
X_train.shape,X_val.shape

((28000, 182), (12000, 182))

We will now apply One Hot Encoding for the the categorical columns in the X_train and X_val

In [15]:
# 1. Initialize the encoder (drop='first' mimics what you used before)
ohe = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform ONLY on the categorical training columns
train_encoded_array = ohe.fit_transform(X_train[cat_cols])
# 3. Transform ONLY on the categorical validation columns
val_encoded_array = ohe.transform(X_val[cat_cols])

# 4. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
X_train_encoded = pd.DataFrame(
    train_encoded_array,
    columns=encoded_feature_names,
    index=X_train.index
)

X_val_encoded = pd.DataFrame(
    val_encoded_array,
    columns=encoded_feature_names,
    index=X_val.index
)

X_train_encoded.head()


,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
39167,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
17329,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
20411,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
19056,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1001,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [16]:
# 3. Drop the old unencoded categorical columns from X_train and X_val
# and concatenate the new OHE columns alongside the numeric ones
X_train_ohe = pd.concat(
    [X_train.drop(columns=cat_cols), X_train_encoded],
    axis=1
)

X_val_ohe = pd.concat(
    [X_val.drop(columns=cat_cols), X_val_encoded],
    axis=1
)

X_train_ohe.head()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m

### Inspect for Feature Importance to the target variable using the correlation
We will apply correlation to see the linear relationship between each feature and the target

In [17]:
# 1. Calculate correlation with y_train and convert the resulting Series to a DataFrame
correlation_df = X_train_ohe.corrwith(y_train).to_frame(name='Correlation')

# 2. Sort by absolute correlation to see the most impactful features at the top
correlation_df = (
    correlation_df
    .assign(Abs_Correlation=correlation_df['Correlation'].abs())
    .sort_values(by='Abs_Correlation', ascending=False)
    .drop(columns=['Abs_Correlation'])
)

# Display the DataFrame
correlation_df

,Correlation
m6_daily_avg_bal,0.081456
m5_daily_avg_bal,0.080179
m2_deposit_agents,-0.077135
m4_daily_avg_bal,0.076991
m1_deposit_agents,-0.075848
...,...
m5_merchantpay_volume,0.001420
region_Lake Zone (Northwest),-0.001088
region_Northern Highlands,-0.000969
m1_mm_send_recipients,0.000900


The Linear relationships are very weak this could mean the relationships are non-linear

### CHECK FOR COLLINEARITY (within the features)
We will check for collinearity amongst the numerical variables.

In [18]:
## Correlation function that find the linear relationship between features using a given threshold to identify collinear relationship

def correlation(dataset, threshold):
    col_corr = set()
    col_pair = set()

    corr_matrix = dataset.corr()

    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                colname1 = corr_matrix.columns[i]
                colname2 = corr_matrix.columns[j]

                col_pair.add((colname1, colname2, corr_matrix.iloc[i, j]))
                col_corr.add(colname1)

    return col_corr, col_pair

In [19]:
col_corr, col_pair = correlation(X_train_ohe[num_cols], 0.7)
col_pair

{('m1_daily_avg_bal', 'arpu', np.float64(0.7308313069940098)),
 ('m1_deposit_highest_amount',
  'm1_deposit_volume',
  np.float64(0.7160150518210067)),
 ('m1_merchantpay_highest_amount',
  'm1_merchantpay_total_value',
  np.float64(0.732876488427667)),
 ('m1_merchantpay_highest_amount',
  'm1_merchantpay_volume',
  np.float64(0.7235033996398849)),
 ('m1_merchantpay_total_value',
  'm1_merchantpay_volume',
  np.float64(0.7073925478210842)),
 ('m1_mm_send_highest_amount', 'arpu', np.float64(0.730903847898394)),
 ('m1_mm_send_highest_amount',
  'm1_mm_send_volume',
  np.float64(0.7235369872158953)),
 ('m1_paybill_highest_amount',
  'm1_paybill_volume',
  np.float64(0.7222304702364004)),
 ('m1_received_highest_amount', 'arpu', np.float64(0.7228410618552629)),
 ('m1_received_highest_amount',
  'm1_received_volume',
  np.float64(0.7263052828314429)),
 ('m1_transfer_from_bank_highest_amount',
  'm1_transfer_from_bank_volume',
  np.float64(0.7794793649706883)),
 ('m1_withdraw_highest_amount', 

These are the collinear pairs. We will have to drop some of the pairs systematically in a way that we compare with the relationship with the target. 

In [20]:
threshold = 0.7

# Strength of each feature's relationship with the target
target_corr = X_train_ohe[num_cols].corrwith(y_train).abs()

# Pairwise feature correlations (upper triangle only, so each pair appears once)
corr = X_train_ohe[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

pairs = (
    upper.stack()
    .rename("pair_corr")
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b"})
)
pairs = pairs[pairs["pair_corr"] > threshold].sort_values("pair_corr", ascending=False)

# For each pair, drop the feature with the weaker target correlation
to_drop, log = set(), []
for _, r in pairs.iterrows():
    a, b = r["feat_a"], r["feat_b"]
    if a in to_drop or b in to_drop:      # pair already resolved by an earlier drop
        continue
    drop, keep = (a, b) if target_corr[a] < target_corr[b] else (b, a)
    to_drop.add(drop)
    log.append({
        "dropped": drop,
        "kept": keep,
        "pair_corr": round(r["pair_corr"], 3),
        "dropped_target_corr": round(target_corr[drop], 4),
        "kept_target_corr": round(target_corr[keep], 4),
        "gap": round(target_corr[keep] - target_corr[drop], 4),
    })

drop_log = pd.DataFrame(log)

# Pairs where the two features are nearly tied on target correlation: review by hand
close_calls = drop_log[drop_log["gap"] < 0.01]

drop_log

,dropped,kept,pair_corr,dropped_target_corr,kept_target_corr,gap
0,m1_transfer_from_bank_volume,m1_transfer_from_bank_highest_amount,0.779,0.0106,0.0159,0.0053
1,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_volume,0.776,0.0069,0.0256,0.0187
2,m2_transfer_from_bank_volume,m2_transfer_from_bank_highest_amount,0.774,0.0120,0.0140,0.0020
3,m5_transfer_from_bank_volume,m5_transfer_from_bank_highest_amount,0.774,0.0142,0.0166,0.0024
4,m6_transfer_from_bank_volume,m6_transfer_from_bank_highest_amount,0.770,0.0079,0.0166,0.0087
5,m4_transfer_from_bank_volume,m4_transfer_from_bank_highest_amount,0.769,0.0153,0.0262,0.0109
6,m2_merchantpay_highest_amount,m2_merchantpay_total_value,0.747,0.0261,0.0278,0.0018
7,m4_merchantpay_highest_amount,m4_merchantpay_total_value,0.741,0.0118,0.0180,0.0062
8,m3_merchantpay_highest_amount,m3_merchantpay_total_value,0.739,0.0202,0.0280,0.0078
9,m3_daily_avg_bal,arpu,0.738,0.0040,0.0295,0.0255


In [21]:
## number of columns to drop
len(to_drop)

51

In [22]:
## convert to list
features_to_drop=list(to_drop)
features_to_drop

['m1_deposit_highest_amount',
 'm2_merchantpay_volume',
 'm2_merchantpay_highest_amount',
 'm2_mm_send_volume',
 'm4_withdraw_highest_amount',
 'm4_mm_send_volume',
 'm4_deposit_volume',
 'm3_daily_avg_bal',
 'm6_received_volume',
 'm5_withdraw_volume',
 'm3_merchantpay_highest_amount',
 'm4_merchantpay_highest_amount',
 'm5_merchantpay_volume',
 'arpu',
 'm1_paybill_volume',
 'm4_transfer_from_bank_volume',
 'm1_merchantpay_volume',
 'm2_paybill_volume',
 'm3_transfer_from_bank_highest_amount',
 'm2_deposit_highest_amount',
 'm6_merchantpay_volume',
 'm2_withdraw_highest_amount',
 'm1_merchantpay_total_value',
 'm6_mm_send_volume',
 'm5_deposit_volume',
 'm3_deposit_highest_amount',
 'm5_paybill_volume',
 'm1_mm_send_highest_amount',
 'm1_received_highest_amount',
 'm6_withdraw_highest_amount',
 'm5_mm_send_volume',
 'm4_paybill_volume',
 'm5_transfer_from_bank_volume',
 'm4_received_volume',
 'm3_merchantpay_volume',
 'm6_deposit_volume',
 'm6_merchantpay_total_value',
 'm3_paybill_v

In [23]:
X_train_ohe_adj=X_train_ohe.drop(columns=features_to_drop)
X_train_ohe_adj.head()

,age,x_90_d_activity_rate,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_total_value,m2_merchantpay_merchants,m3_merchantpay_total_value,m3_merchantpay_merchants,m4_merchantpay_total_value,m4_merchantpay_merchants,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_banks,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_recipients,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_recipients,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_senders,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_agents,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_agents,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m6_withdraw_total_value,m6_withdraw_agents,m1_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
39167,43,0.1294,825.95,577.40,1.0,0.00,0.00,0.0,0.00,0.00,0.0,296.16,249.19,6.0,0.00,0.00,0.0,0.00,0.00,0.0,203.71,13.0,0.00,0.0,0.00,0.0,0.00,0.0,525.40,2.0,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0,0.00,0.0,0.00,0.00,0.0,5,13565.81,16.0,6742.73,942.96,14.0,5218.43,861.09,12.0,0.00,0.00,0.0,8,4452.87,6.0,0,0.00,0.0,4,2026.40,18.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0,0.00,0.0,0,0.00,0.0,6,12330.30,1.0,4390.58,954.39,5.0,10100.51,3227.33,2.0,18535.26,1244.31,2.0,4,2552.59,1.0,2,1329.77,3.0,3,7784.15,5.0,6,12554.69,4.0,0.00,0.00,0.0,5,4215.51,5.0,893.66,2891.25,1743.36,2008.44,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
17329,23,0.6899,34728.75,5932.32,8.0,26547.15,3762.91,3.0,0.00,0

In [24]:
X_val_ohe_adj=X_val_ohe.drop(columns=features_to_drop)


In [25]:
X_val_ohe_adj.shape, X_train_ohe_adj.shape

((12000, 138), (28000, 138))

## Grid Search

We will apply grid search to identify the best parameters to get optimal result. 

In [26]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score,log_loss
from sklearn.linear_model import LogisticRegression


In [27]:
param_distributions = {
        "depth": [4, 6],
        "learning_rate": [0.05, 0.1],
        "iterations": [200, 400],
        "auto_class_weights": ["None", "Balanced"],
        "l2_leaf_reg": [1, 3, 5], # Adds essential L2 regularization to prevent overfitting
        "random_strength": [1, 5]
}

In [28]:
from catboost import CatBoostClassifier

In [ ]:
## Run grid
RANDOM_STATE=42
model=CatBoostClassifier(random_state=RANDOM_STATE, thread_count=-1, verbose=0)

## Scale the data
scaler=StandardScaler()
X_train_ohe_adj_scaled=scaler.fit_transform(X_train_ohe_adj)
X_val_ohe_adj_scaled=scaler.transform(X_val_ohe_adj)

random=RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1
    )
random.fit(X_train_ohe_adj_scaled,y_train) ## Note we are not using oversampled data
best_parameters=random.best_params_

print(f'______________ Best Params for Catboost Regression _________________')
print(best_parameters)

c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 96 is smaller than n_iter=100. Running 96 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 96 candidates, totalling 288 fits
______________ Best Params for Logistic Regression _________________
{'random_strength': 1, 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 400, 'depth': 6, 'auto_class_weights': 'None'}


In [29]:
# best_parameters={'random_strength': 1, 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 400, 'depth': 6, 'auto_class_weights': 'None'}

best_parameters={'random_strength': 1, 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 400, 'depth': 6, 'auto_class_weights': 'Balanced'}

In [30]:
from sklearn.base import clone
import pandas as pd
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)


In [31]:
RANDOM_STATE=42
model=CatBoostClassifier(random_state=RANDOM_STATE, thread_count=-1, verbose=0)


In [32]:
## Scale the data
scaler=StandardScaler()
X_train_ohe_adj_scaled=scaler.fit_transform(X_train_ohe_adj)
X_val_ohe_adj_scaled=scaler.transform(X_val_ohe_adj)

In [33]:
print(f"Catboost training")

# fresh copy of the estimator with the tuned params applied
model_obj = clone(model).set_params(**best_parameters)
model_obj.fit(X_train_ohe_adj_scaled, y_train)

# probabilities and labels
train_prob = model_obj.predict_proba(X_train_ohe_adj_scaled)[:, 1]
val_prob = model_obj.predict_proba(X_val_ohe_adj_scaled)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)
train_pred = (train_prob >= 0.5).astype(int)

train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)
rep = classification_report(y_val, val_pred, output_dict=True)

print(f"Train ROC-AUC: {train_auc} | Validation ROC-AUC: {val_auc}")
print(f"Train accuracy: {accuracy_score(y_train, train_pred)} | "
        f"Validation accuracy: {accuracy_score(y_val, val_pred)}")
## Log-loss for the predicted classes
print(f"Train prediction log_loss: {log_loss(y_train, train_pred)} | "
        f"Validation prediction log_loss: {log_loss(y_val, val_pred)}")
## Log-loss for the predicted probability
print(f"Train probability log loss: {log_loss(y_train, train_prob)} | Val probability log loss: {log_loss(y_val, val_prob)}")

print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))
print("\n" + "-" * 60 + "\n")


Catboost training
Train ROC-AUC: 0.9708121348539417 | Validation ROC-AUC: 0.8350781045751634
Train accuracy: 0.8961071428571429 | Validation accuracy: 0.8155
Train prediction log_loss: 3.7446781324622074 | Validation prediction log_loss: 6.650054050292114
Train probability log loss: 0.3118389342274386 | Val probability log loss: 0.41114820588164686
              precision    recall  f1-score   support

           0       0.93      0.85      0.89     10200
           1       0.42      0.63      0.51      1800

    accuracy                           0.82     12000
   macro avg       0.68      0.74      0.70     12000
weighted avg       0.85      0.82      0.83     12000

[[8646 1554]
 [ 660 1140]]

------------------------------------------------------------



The prediction of the minority class is low especially the recall. Let us inspect the performance with a better threshold. The log-loss for predicted value is based on the natural threshold selected at 0.5. Our focus should be on the log-loss of the prredicted probability which is 
`Train probability log loss: 0.2112296642101178 | Val probability log loss: 0.31003831971670026`

In [34]:
from sklearn.metrics import precision_recall_curve

In [35]:
# 7. Evaluate with best threshold

# 8. Tune threshold on validation (F1 for class 1)
prec, rec, thr = precision_recall_curve(y_val, val_prob)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
best_thr = thr[np.argmax(f1)]
print("Best threshold:", round(float(best_thr), 3))

val_pred = (val_prob >= best_thr).astype(int)

print("performance and classification_report with best threshold")
print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

Best threshold: 0.624
performance and classification_report with best threshold
              precision    recall  f1-score   support

           0       0.91      0.92      0.92     10200
           1       0.54      0.51      0.52      1800

    accuracy                           0.86     12000
   macro avg       0.73      0.72      0.72     12000
weighted avg       0.86      0.86      0.86     12000

[[9422  778]
 [ 888  912]]


Improved recall.

## Create submission 

In [36]:
SUBMISSION_TARGET = "Target"

# Create a sample submission template directly from the test IDs
submission = pd.DataFrame({
    "ID": test["ID"],
    SUBMISSION_TARGET: 0.5
})


In [37]:
test_clean=test.drop(columns=['ID'])

In [38]:
# Apply one hot encoding
# 2. Fit transform ONLY on the categorical training columns
test_encoded_array = ohe.transform(test_clean[cat_cols])

# 3. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
test_encoded = pd.DataFrame(
    test_encoded_array,
    columns=encoded_feature_names,
    index=test_clean.index
)

test_encoded.head()

,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [39]:
test_ohe = pd.concat(
    [test_clean.drop(columns=cat_cols), test_encoded],
    axis=1
)

test_ohe.head()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m

In [40]:
## Adjust the number of features
test_ohe_adj=test_ohe.drop(columns=features_to_drop)
test_ohe_adj.head()

,age,x_90_d_activity_rate,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_total_value,m2_merchantpay_merchants,m3_merchantpay_total_value,m3_merchantpay_merchants,m4_merchantpay_total_value,m4_merchantpay_merchants,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_banks,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_recipients,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_recipients,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_senders,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_agents,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_agents,m5_withdraw_total_value,m5_withdraw_highest_amount,m5_withdraw_agents,m6_withdraw_volume,m6_withdraw_total_value,m6_withdraw_agents,m1_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
0,47,0.6905,0.00,0.00,0.0,0.00,0.00,0.0,21341.77,5938.05,8.0,7831.85,1526.06,3.0,28295.28,6259.96,3.0,0.0,0.0,0.0,1982.82,2.0,1505.08,11.0,4373.46,4.0,3790.56,5.0,4346.68,9.0,2618.07,11.0,27009.99,11265.21,11.0,0.00,0.00,0.0,4,42572.01,8.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.0,3,11931.62,6.0,37721.15,3275.68,20.0,0,0.00,0.0,25476.62,3207.89,18.0,0.00,0.00,0.0,9137.70,4157.57,1.0,20,65202.88,2.0,13,76219.65,20.0,0,0.00,0.0,0.00,0.00,0.0,49904.10,2781.92,13.0,48561.73,2258.02,4.0,14,109128.91,1.0,13,120493.24,1.0,10,111806.67,3.0,0.00,0.00,0.0,189727.10,4672.05,5.0,262377.52,17235.39,2.0,9,40811.44,5.0,11,62921.82,3.0,0,0.00,0.0,0,0.00,0.0,79416.13,4663.77,3.0,0,0.00,0.0,7272.55,12989.74,11233.97,16461.21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0

In [41]:
## Scale the data
test_ohe_adj_scaled=scaler.transform(test_ohe_adj)
test_ohe_adj_scaled

array([[ 0.1846283 ,  0.81121636, -0.2674084 , ...,  0.97697919,
         1.74601919, -1.22739155],
       [ 0.1846283 ,  0.81121636,  0.4371055 , ...,  0.97697919,
         1.74601919, -1.22739155],
       [ 0.1846283 ,  0.81121636, -0.2674084 , ...,  0.97697919,
         1.74601919, -1.22739155],
       ...,
       [ 0.83737936,  1.21950823, -0.06120475, ...,  0.97697919,
        -0.57273139,  0.81473593],
       [ 0.83737936,  1.21950823, -0.01956956, ...,  0.97697919,
        -0.57273139,  0.81473593],
       [ 0.83737936,  1.21950823,  0.20413674, ...,  0.97697919,
        -0.57273139,  0.81473593]], shape=(30000, 138))

In [42]:
test_prob=model_obj.predict_proba(test_ohe_adj_scaled)[:, 1]

In [43]:
submission[SUBMISSION_TARGET] = test_prob

In [44]:
submission

,ID,Target
0,ID_ED91AC7A28,0.263675
1,ID_29975A9C7E,0.100878
2,ID_F59CE25650,0.078344
3,ID_E875F68D1A,0.025782
4,ID_2902B1C0D0,0.060303
...,...,...
29995,ID_3C7CF79AEE,0.094938
29996,ID_F1651E73C4,0.192279
29997,ID_FD3F0934CA,0.359283
29998,ID_DA6916D6C7,0.097134


just a quick inspection

In [45]:
classification = (test_prob >= 0.5).astype(int)

In [46]:
classification

array([0, 0, 0, ..., 0, 0, 0], shape=(30000,))

In [47]:
from collections import Counter
class_freq=Counter(classification)
class_freq

Counter({np.int64(0): 22996, np.int64(1): 7004})

In [48]:
# Save to CSV
submission.to_csv("submission_Catboost_balanced.csv", index=False)
print("Submission file saved!")


Submission file saved!


lets try submitting the oversampled version